# Preliminaires E1-E3 -- validation du threat model label-flipping

Ce notebook valide empiriquement trois predictions bloquantes du modele theorique
d'attaque backdoor par inversion d'etiquettes en federe :

- **E1** : l'identite `E[g_i] = grad_c + Gbar @ u_i` transfère-t-elle d'un jeu de
  calibration a un shard reel, et le signal reste-t-il au-dessus du bruit minibatch ?
- **E2** : le plafond de rang (`varpi`) laisse-t-il une marge exploitable a
  l'attaquant, ou la politique par classe est-elle trop grossiere ?
- **E3** : une configuration `ubar` unique sert-elle toute la trajectoire
  d'entrainement (one-shot), ou le support de l'attaque optimale tourne-t-il
  d'un checkpoint a l'autre ?

**Politique de reuse** : tout le code de `modules/federated_optimizing_trigger/utils.py`
et `modules/base_utils/` (matrice de shifts, solveur QP a budget global, chargeurs
de donnees, sharding IID, boucle d'entrainement `mini_train_multi`) est importe
tel quel depuis `prelim_lib.py`, jamais copie ni modifie. `prelim_lib.py` n'ajoute
que ce qui manque reellement : la variante du QP avec plafonds par classe, la
realisation dure des masses de flip, un modele lineaire (absent du depot), et les
diagnostics de E1-E3 (SNR, rayon atteignable, fonction d'appui, rank ratio).

**Note de correction (session precedente)** : E1 ne teste PAS la linearite du modele
(l'identite est exacte par construction, quel que soit le modele) -- elle teste
(i) l'implementation, (ii) le transfert calibration -> shard, (iii) le rapport
signal/bruit minibatch. Voir la cellule Verdict d'E1.

**Deviation de perimetre** : ce depot n'a ni regression logistique ni MNIST/Fashion-MNIST.
La config `linear` est donc une regression logistique multinomiale brute sur
CIFAR-10 aplati (3072 -> 10), comme prevu par la clause de repli de la session
precedente ; la config `cnn` utilise `r32p` (ResNet-32, le plus petit CNN du depot).


In [9]:
%pip install torch torchvision torchaudio osqp tqdm toml

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
Using cached toml-0.10.2-py2.py3-none-any.whl (16 kB)
  DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621
DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew

In [ ]:
import os, sys, math, itertools, json, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("."))
import prelim_lib as pl

# modules/base_utils/datasets.py's PATH dict is relative to cwd ("./data/...");
# chdir to the repo root so it resolves to the canonical data/ dir shared with
# the rest of the pipeline, instead of creating a second copy under prelim/.
os.chdir(pl._FLIP_ROOT)
print("cwd:", os.getcwd())

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
ARTIFACT_DIR = os.path.join(pl._FLIP_ROOT, "prelim", "artifacts")
CKPT_DIR = os.path.join(ARTIFACT_DIR, "ckpt")
FIG_DIR = os.path.join(ARTIFACT_DIR, "figs")
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

METRICS = []
def record(config, seed, checkpoint, beta, experiment, metric, value):
    METRICS.append(dict(config=config, seed=seed, checkpoint=checkpoint,
                         beta=beta, experiment=experiment, metric=metric,
                         value=float(value)))

print("device:", DEVICE)


cwd: /Users/martinbeaufils/Downloads/broadflip_repo/FLIP
device: mps


In [11]:
# ---- choisir UNE config et relancer le notebook a partir d'ici ------------
CONFIG_NAME = "linear"   # "linear" | "cnn"

DATASET_FLAG = "cifar"
N_CLASSES = 10
MODEL_FLAG = "linear" if CONFIG_NAME == "linear" else "r32p"

N_B, N_P, F = 10, 3, 3          # grille reelle de la campagne (confirme par l'utilisateur)
GAMMA = N_P / N_B                # 0.3
BETAS = [0.01, 0.03, 0.10]
Y_SOURCE, Y_TARGET = 9, 4        # confirme par experiments/.../opt_trigger/config.toml
SEEDS = [0, 1, 2]
TRIGGERS = ["identity", "stripe"]
CKPT_NAMES = ["begin", "mid", "end"]

if CONFIG_NAME == "linear":
    TRAIN_EPOCHS = 8
    TRAIN_BATCH = 256
    MAX_TRAIN = None      # dataset complet (n=50000), coherent avec l'exemple
                           # beta = N_flip/n = 5000/50000 = 0.10 donne par l'utilisateur
elif CONFIG_NAME == "cnn":
    TRAIN_EPOCHS = 4
    TRAIN_BATCH = 128
    MAX_TRAIN = 10000      # cf. correction : 10000 si le temps le permet (sinon repasser a 5000)
else:
    raise ValueError(CONFIG_NAME)

print(f"CONFIG_NAME={CONFIG_NAME} model={MODEL_FLAG} n_b={N_B} n_p={N_P} "
      f"gamma={GAMMA:.3f} betas={BETAS} source->target={Y_SOURCE}->{Y_TARGET}")


CONFIG_NAME=linear model=linear n_b=10 n_p=3 gamma=0.300 betas=[0.01, 0.03, 0.1] source->target=9->4


## Entrainement propre court -> 3 checkpoints

Seul entrainement de la session : quelques epoques sur donnees propres,
checkpoints sauvegardes a begin/mid/end sous `prelim/artifacts/ckpt/`
(jamais sous `out/checkpoints/`). Reutilise `mini_train_multi`
(`modules/base_utils/util.py`) tel quel, avec un seul "worker" (le jeu propre
entier) et `agg_method="mean"` -- ce qui degenere exactement en SGD standard.


In [12]:
t0 = time.time()
torch.manual_seed(SEEDS[0])
model_init = pl.build_model(MODEL_FLAG, N_CLASSES, DEVICE)

ckpt_paths = pl.train_clean_checkpoints(
    model_init, DATASET_FLAG, N_CLASSES, DEVICE,
    epochs=TRAIN_EPOCHS, batch_size=TRAIN_BATCH,
    ckpt_dir=CKPT_DIR, tag=CONFIG_NAME, test_pct=0.1,
    seed=SEEDS[0], max_train=MAX_TRAIN,
)
print(f"entraine en {time.time()-t0:.1f}s ->", ckpt_paths)


100.0%
  0%|          | 0/200000 [00:00<?, ?it/s]/opt/homebrew/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
  0%|          | 0/200000 [00:00<?, ?it/s]


PicklingError: Can't pickle <function <lambda> at 0x159c2bc10>: attribute lookup <lambda> on modules.base_utils.datasets failed

In [ ]:
N_PER_CLASS_CALIB = 512
class_samples_raw = pl.get_class_conditional_samples(
    DATASET_FLAG, N_CLASSES, N_PER_CLASS_CALIB, DEVICE)
print({y: tuple(t.shape) for y, t in class_samples_raw.items()})

def load_ckpt(name):
    m = pl.build_model(MODEL_FLAG, N_CLASSES, DEVICE)
    m.load_state_dict(torch.load(ckpt_paths[name], map_location=DEVICE))
    # .eval() matches the pipeline's own convention around
    # compute_expected_flip_gradients: federated_optimizing_trigger/run_module.py
    # does `expert_models[k].to(device).eval()` before calling it. Using eval()
    # everywhere (Gbar/grad_c/grad_bd AND worker_gradient) avoids conflating
    # BatchNorm batch-statistics noise (r32p) with the label-flip signal we're
    # actually trying to measure.
    m.eval()
    return m

CACHE = {}
for name in CKPT_NAMES:
    m = load_ckpt(name)
    Gbar, grad_c, pi, pairs, col_index, Q = pl.class_conditional_shifts(
        m, class_samples_raw, DATASET_FLAG, N_CLASSES, DEVICE,
        loss_fn=pl.clf_loss, model_flag=None)
    grad_bd = {t: pl.compute_grad_bd(m, pl.clf_loss, class_samples_raw, Y_SOURCE, Y_TARGET,
                                      DATASET_FLAG, DEVICE, model_flag=None, trigger=t)
               for t in TRIGGERS}
    CACHE[name] = dict(Gbar=Gbar, grad_c=grad_c, pi=pi, pairs=pairs,
                        col_index=col_index, Q=Q, grad_bd=grad_bd)
    del m
    print(name, "Gbar", tuple(Gbar.shape), "||grad_c||=%.4g" % float(grad_c.norm()))

def v_of(name, beta, trigger="identity"):
    """v = grad_p - grad_c = lam*(grad_bd - grad_c), lam = beta (convention de session)."""
    c = CACHE[name]
    return beta * (c["grad_bd"][trigger] - c["grad_c"])


## E1 -- Carte de biais : implementation, transfert, signal/bruit (BLOQUANT)

**Recadrage (correction de session)** : l'identite `E[g_i] = grad_c + Gbar @ u_i`
est exacte par construction pour n'importe quel modele -- ce n'est PAS un test de
linearite. E1 teste trois choses distinctes, et le verdict est ecrit dans ces
termes :

1. **implementation** (signes, ponderation par `pi[y]`, ordre des colonnes) ;
2. **transfert** du `Gbar` estime sur le jeu de calibration vers un shard reel
   (proportions de classe et gradients conditionnels propres au shard) ;
3. **rapport signal/bruit** minibatch (`SNR = ||Gbar@u|| / (sigma_i/sqrt(|B|))`).

Si le cosinus est nettement < 0.99 avec le `Gbar` **recalcule sur le shard**,
la carte de biais est fausse -- tout le cadre theorique tombe. Si le SNR << 1
aux budgets realistes, la deviation d'attaque est noyee dans le bruit minibatch.

**Perimetre retenu pour tenir le budget de 10 minutes** : les 6 configurations
`u` sont comparees aux 3 checkpoints avec `beta=0.10` (le budget le plus
informatif) ; la robustesse aux graines est verifiee sur la configuration (a)
au checkpoint `end` sur les 3 graines ; le balayage de taille de batch et la
grille SNR(beta, B) sont faits au checkpoint `end`, configuration (a).


In [ ]:
raw_dataset = pl.get_raw_clean_dataset(DATASET_FLAG, train=True)
all_targets = np.array(raw_dataset.dataset.targets)

def build_u_configs(pi, gamma, beta, y_source, y_target, pairs, Q, c_qp, seed):
    """6 configurations u dans U_loc, protocole E1 (a)-(f)."""
    rng = np.random.RandomState(seed)
    P = len(pairs)
    budget_loc = beta / gamma
    classes = sorted(set(y for y, z in pairs))

    configs = {}

    # (a) concentree sur (y_source, y_target)
    u = {p: 0.0 for p in pairs}
    u[(y_source, y_target)] = min(budget_loc, pi[y_source])
    configs["a_source_target"] = u

    # (b) concentree sur une autre paire
    y2 = classes[(classes.index(y_source) + 3) % len(classes)]
    z2 = classes[(classes.index(y_target) + 3) % len(classes)]
    if z2 == y2:
        z2 = classes[(classes.index(z2) + 1) % len(classes)]
    u = {p: 0.0 for p in pairs}
    u[(y2, z2)] = min(budget_loc, pi[y2])
    configs["b_other_pair"] = u

    # (c) uniforme sur toutes les paires, puis clip pour respecter les plafonds par classe
    per_pair = budget_loc / P
    u = {p: per_pair for p in pairs}
    for y in classes:
        idxs = [p for p in pairs if p[0] == y]
        tot = sum(u[p] for p in idxs)
        if tot > pi[y]:
            scale = pi[y] / tot
            for p in idxs:
                u[p] *= scale
    configs["c_uniform"] = u

    # (d) sortie du QP (scope=aggregate, capacity=True) -> u_i = ubar*/gamma
    ubar_star = pl.solve_qp(Q, c_qp, beta, pi, gamma, pairs,
                             scope="aggregate", capacity=True).numpy()
    # correction A: verification numerique du lien u_i = ubar/gamma
    u_d = {p: ubar_star[i] / gamma for i, p in enumerate(pairs)}
    assert abs(sum(u_d.values()) * gamma - ubar_star.sum()) < 1e-6, \
        "lien u_i = ubar/gamma viole numeriquement"
    configs["d_qp"] = u_d

    # (e), (f): points aleatoires faisables dans U_loc (projection via solve_qp,
    # scope="local" -- reutilise directement la meme machinerie OSQP)
    for tag in ["e_random1", "f_random2"]:
        raw_pt = rng.rand(P) * (budget_loc / P) * 2
        proj = pl.solve_qp(np.eye(P), raw_pt, beta, pi, gamma, pairs,
                            scope="local", capacity=True).numpy()
        configs[tag] = {p: proj[i] for i, p in enumerate(pairs)}

    # sanity: verifier l'appartenance a U_loc
    for tag, u in configs.items():
        total = sum(u.values())
        assert total <= budget_loc + 1e-5, f"{tag}: budget viole ({total} > {budget_loc})"
        for y in classes:
            s = sum(u[p] for p in pairs if p[0] == y)
            assert s <= pi[y] + 1e-5, f"{tag}: plafond classe {y} viole ({s} > {pi[y]})"
    return configs


E1_BETA = 0.10
rows_e1 = []

for name in CKPT_NAMES:
    m = load_ckpt(name)
    c = CACHE[name]
    Gbar, grad_c, pi, pairs = c["Gbar"], c["grad_c"], c["pi"], c["pairs"]
    v = v_of(name, E1_BETA, "identity")
    c_qp = (Gbar.T @ v).numpy().astype(np.float64)

    configs = build_u_configs(pi, GAMMA, E1_BETA, Y_SOURCE, Y_TARGET, pairs, c["Q"], c_qp, seed=SEEDS[0])

    shard_idx_list = pl.shard_indices(raw_dataset, N_B, seed=SEEDS[0])
    worker_shard = shard_idx_list[0]
    shard_true_targets = all_targets[worker_shard]

    for tag, u in configs.items():
        rng = np.random.RandomState(SEEDS[0])
        poisoned_targets, u_real = pl.flip_masses_to_labels(shard_true_targets, u, pairs, rng)

        u_real_vec = torch.tensor([u_real[p] for p in pairs], dtype=torch.float32)
        Gbar_u = Gbar @ u_real_vec
        pred = grad_c + Gbar_u

        g_emp = pl.worker_gradient(m, pl.clf_loss, raw_dataset, worker_shard, poisoned_targets,
                                    DATASET_FLAG, None, batch_size=1024, device=DEVICE)

        err_rel = float((g_emp - pred).norm() / (Gbar_u.norm() + 1e-12))
        cos = float(torch.nn.functional.cosine_similarity(
            (g_emp - grad_c).unsqueeze(0), Gbar_u.unsqueeze(0)).item())
        record(CONFIG_NAME, SEEDS[0], name, E1_BETA, "E1", f"relerr_calib__{tag}", err_rel)
        record(CONFIG_NAME, SEEDS[0], name, E1_BETA, "E1", f"cos_calib__{tag}", cos)

        # Gbar recalcule SUR LE SHARD (memes indices) -- separe erreur d'estimation / de modele
        shard_samples_raw = {}
        for y in np.unique(shard_true_targets):
            idx_y = worker_shard[shard_true_targets == y]
            xs = torch.stack([raw_dataset[int(i)][0] for i in idx_y]).to(DEVICE)
            shard_samples_raw[int(y)] = xs
        Gbar_s, grad_c_s, pi_s, pairs_s, col_s, Q_s = pl.class_conditional_shifts(
            m, shard_samples_raw, DATASET_FLAG, N_CLASSES, DEVICE, loss_fn=pl.clf_loss, model_flag=None)
        u_real_vec_s = torch.tensor([u_real.get(p, 0.0) for p in pairs_s], dtype=torch.float32)
        Gbar_u_s = Gbar_s @ u_real_vec_s
        pred_s = grad_c_s + Gbar_u_s
        err_rel_s = float((g_emp - pred_s).norm() / (Gbar_u_s.norm() + 1e-12))
        cos_s = float(torch.nn.functional.cosine_similarity(
            (g_emp - grad_c_s).unsqueeze(0), Gbar_u_s.unsqueeze(0)).item())
        record(CONFIG_NAME, SEEDS[0], name, E1_BETA, "E1", f"relerr_shard__{tag}", err_rel_s)
        record(CONFIG_NAME, SEEDS[0], name, E1_BETA, "E1", f"cos_shard__{tag}", cos_s)

        rows_e1.append(dict(checkpoint=name, config=tag, err_rel=err_rel, cos=cos,
                             err_rel_shard=err_rel_s, cos_shard=cos_s,
                             n_classes_in_shard_Gbar=len(pairs_s)))
    del m

df_e1 = pd.DataFrame(rows_e1)
df_e1


In [ ]:
# --- robustesse aux graines (config a, checkpoint end) ---------------------
name = "end"
m = load_ckpt(name)
c = CACHE[name]
Gbar, grad_c, pi, pairs = c["Gbar"], c["grad_c"], c["pi"], c["pairs"]

seed_rows = []
for seed in SEEDS:
    shards = pl.shard_indices(raw_dataset, N_B, seed=seed)
    worker_shard = shards[0]
    shard_true_targets = all_targets[worker_shard]
    u_a = {p: 0.0 for p in pairs}
    u_a[(Y_SOURCE, Y_TARGET)] = min(E1_BETA / GAMMA, pi[Y_SOURCE])
    rng = np.random.RandomState(seed)
    poisoned_targets, u_real = pl.flip_masses_to_labels(shard_true_targets, u_a, pairs, rng)
    u_real_vec = torch.tensor([u_real[p] for p in pairs], dtype=torch.float32)
    Gbar_u = Gbar @ u_real_vec
    pred = grad_c + Gbar_u
    g_emp = pl.worker_gradient(m, pl.clf_loss, raw_dataset, worker_shard, poisoned_targets,
                                DATASET_FLAG, None, batch_size=1024, device=DEVICE)
    err_rel = float((g_emp - pred).norm() / (Gbar_u.norm() + 1e-12))
    cos = float(torch.nn.functional.cosine_similarity(
        (g_emp - grad_c).unsqueeze(0), Gbar_u.unsqueeze(0)).item())
    seed_rows.append(dict(seed=seed, err_rel=err_rel, cos=cos))
    record(CONFIG_NAME, seed, name, E1_BETA, "E1", "seed_robustness_err_rel__a", err_rel)
    record(CONFIG_NAME, seed, name, E1_BETA, "E1", "seed_robustness_cos__a", cos)

df_seeds = pd.DataFrame(seed_rows)
print(df_seeds)
print(f"cos: mean={df_seeds.cos.mean():.4f} std={df_seeds.cos.std():.4f}")

# --- balayage taille de batch + grille SNR(beta, B) -------------------------
BATCH_SIZES = [64, 256, 1024]
N_MB_DRAWS = 6

shards0 = pl.shard_indices(raw_dataset, N_B, seed=SEEDS[0])
worker_shard = shards0[0]
shard_true_targets = all_targets[worker_shard]

sweep_rows = []
for beta in BETAS:
    u_a = {p: 0.0 for p in pairs}
    u_a[(Y_SOURCE, Y_TARGET)] = min(beta / GAMMA, pi[Y_SOURCE])
    rng = np.random.RandomState(SEEDS[0])
    poisoned_targets, u_real = pl.flip_masses_to_labels(shard_true_targets, u_a, pairs, rng)
    u_real_vec = torch.tensor([u_real[p] for p in pairs], dtype=torch.float32)
    Gbar_u = Gbar @ u_real_vec
    pred = grad_c + Gbar_u

    g_full = pl.worker_gradient(m, pl.clf_loss, raw_dataset, worker_shard, poisoned_targets,
                                 DATASET_FLAG, None, batch_size=2048, device=DEVICE)
    err_full = float((g_full - pred).norm() / (Gbar_u.norm() + 1e-12))
    sweep_rows.append(dict(beta=beta, batch_size=len(worker_shard), err_rel=err_full, snr=np.nan))
    record(CONFIG_NAME, SEEDS[0], name, beta, "E1", "sweep_err_rel__B=full", err_full)

    for B in BATCH_SIZES:
        samples = pl.minibatch_gradient_samples(
            m, pl.clf_loss, raw_dataset, worker_shard, poisoned_targets,
            batch_size=B, n_draws=N_MB_DRAWS, dataset_flag=DATASET_FLAG,
            model_flag=None, device=DEVICE, seed=SEEDS[0])
        # erreur typique d'UN tirage a taille B (pas l'erreur de la moyenne des
        # tirages, qui annulerait artificiellement le bruit qu'on veut mesurer)
        per_draw_err = [float((samples[i] - pred).norm() / (Gbar_u.norm() + 1e-12))
                         for i in range(samples.shape[0])]
        err_mb = float(np.mean(per_draw_err))
        snr_val = pl.snr(Gbar_u, samples)
        sweep_rows.append(dict(beta=beta, batch_size=B, err_rel=err_mb, snr=snr_val))
        record(CONFIG_NAME, SEEDS[0], name, beta, "E1", f"sweep_err_rel__B={B}", err_mb)
        record(CONFIG_NAME, SEEDS[0], name, beta, "E1", f"snr__B={B}", snr_val)

del m
df_sweep = pd.DataFrame(sweep_rows)
df_sweep


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(df_e1.err_rel, df_e1.cos)
axes[0].axhline(0.99, color="r", ls="--", lw=1)
axes[0].set_xlabel("erreur relative"); axes[0].set_ylabel("cosinus")
axes[0].set_title("E1: cos vs erreur rel. (Gbar calibration)")

axes[1].scatter(df_e1.err_rel, df_e1.err_rel_shard)
lims = [0, max(df_e1.err_rel.max(), df_e1.err_rel_shard.max()) * 1.1]
axes[1].plot(lims, lims, "k--", lw=1)
axes[1].set_xlabel("erreur rel. (Gbar calibration)")
axes[1].set_ylabel("erreur rel. (Gbar shard)")
axes[1].set_title("Transfert calibration -> shard")

for beta in BETAS:
    sub = df_sweep[(df_sweep.beta == beta) & (df_sweep.batch_size < len(raw_dataset) // N_B)]
    axes[2].plot(sub.batch_size, sub.err_rel, marker="o", label=f"beta={beta}")
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_xlabel("taille de batch B"); axes[2].set_ylabel("erreur relative")
axes[2].set_title("E1: erreur vs |B| (log-log)")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e1_{CONFIG_NAME}.png"), dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
snr_piv = df_sweep[df_sweep.batch_size < len(raw_dataset) // N_B].pivot(
    index="beta", columns="batch_size", values="snr")
im = ax.imshow(snr_piv.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(snr_piv.columns))); ax.set_xticklabels(snr_piv.columns)
ax.set_yticks(range(len(snr_piv.index))); ax.set_yticklabels(snr_piv.index)
ax.set_xlabel("taille de batch B"); ax.set_ylabel("beta")
for i in range(snr_piv.shape[0]):
    for j in range(snr_piv.shape[1]):
        ax.text(j, i, f"{snr_piv.values[i, j]:.2f}", ha="center", va="center", color="w")
plt.colorbar(im)
plt.title("SNR(beta, B) -- checkpoint=end, config=(a)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e1_snr_{CONFIG_NAME}.png"), dpi=130)
plt.show()


### Verdict E1

*(a remplir apres execution : cosinus min/median sur `df_e1` (calibration ET
shard), pente mesuree sur le tracé log-log erreur vs |B|, valeurs de SNR aux
budgets realistes, robustesse aux graines. Conclusion go/no-go.)*


## E2 -- Geometrie et scalaires de regime (BLOQUANT)

**Rappel (correction de session)** : pour la config `linear` (softmax lineaire),
`Gbar` a une structure de rang exactement `C(C-1)` generiquement (produit
externe `(e_y - e_z) x_bar_y^T`) -- `varpi` et `alpha_tilde_star` y seront donc
atypiquement favorables. **La config `linear` valide l'implementation ; seule
la ligne `r32p` (config `cnn`) decide du verdict d'E2.** Ne pas conclure
go/no-go avant d'avoir execute la config `cnn`.

`alpha_tilde_star` (NNLS sans budget) ne demande pas de nouveau solveur : c'est
`project_gradient` avec un budget enorme -- `dist_to_cone` verifie que la
contrainte est bien inactive a l'optimum et alerte sinon.


In [ ]:
d = CACHE["end"]["Gbar"].shape[0]
rows_e2 = []

for name in CKPT_NAMES:
    c = CACHE[name]
    Gbar, grad_c, pi, pairs, Q = c["Gbar"], c["grad_c"], c["pi"], c["pairs"], c["Q"]
    eff_rank = pl.effective_rank(Q)
    baseline = eff_rank / d

    for beta in BETAS:
        v = v_of(name, beta, "identity")
        v_norm = float(v.norm())
        v_norm_sq = v_norm ** 2
        c_qp = (Gbar.T @ v).numpy().astype(np.float64)

        radius = pl.reachable_radius(Gbar, beta, pi, GAMMA, pairs)
        varsigma = radius["varsigma"]
        rho = beta * varsigma
        v_hat = v_norm / rho if rho > 0 else float("inf")

        varpi = pl.rank_ratio(Q, c_qp, v_norm_sq) if v_norm_sq > 0 else float("nan")
        dist2, alpha_tilde_star, _ = pl.dist_to_cone(Q, c_qp, v_norm_sq, pairs)

        grad_c_norm = float(grad_c.norm())
        ratio = radius["lower_ascent"] / grad_c_norm if grad_c_norm > 0 else float("inf")
        Theta = math.asin(ratio) if ratio <= 1 else float("nan")

        grad_p = grad_c + v
        cos_gp_gc = float(torch.nn.functional.cosine_similarity(
            grad_p.unsqueeze(0), grad_c.unsqueeze(0)).item())
        angle_gp_gc = math.degrees(math.acos(max(-1.0, min(1.0, cos_gp_gc))))

        s_beta = beta / (GAMMA * min(pi.values()))

        row = dict(checkpoint=name, beta=beta, varsigma=varsigma, rho=rho,
                   radius_upper=radius["upper"], radius_lower_simple=radius["lower_simple"],
                   radius_lower_ascent=radius["lower_ascent"], v_norm=v_norm, v_hat=v_hat,
                   varpi=varpi, baseline=baseline,
                   alpha_tilde_star=alpha_tilde_star,
                   sqrt_varpi=math.sqrt(varpi) if varpi >= 0 else float("nan"),
                   grad_c_norm=grad_c_norm, Theta_rad=Theta, angle_gp_gc_deg=angle_gp_gc,
                   cos_gp_gc=cos_gp_gc, s_beta=s_beta)
        rows_e2.append(row)
        for k2, v2 in row.items():
            if k2 in ("checkpoint", "beta"):
                continue
            record(CONFIG_NAME, None, name, beta, "E2", k2, v2)

df_e2 = pd.DataFrame(rows_e2)
df_e2


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

eigvals = np.linalg.eigvalsh(CACHE["end"]["Q"])[::-1]
axes[0, 0].plot(np.clip(eigvals, 1e-12, None), marker="o", ms=3)
axes[0, 0].set_yscale("log")
axes[0, 0].set_title("Spectre des valeurs propres de Q (checkpoint=end)")
axes[0, 0].set_xlabel("indice"); axes[0, 0].set_ylabel("valeur propre")

sub = df_e2[df_e2.beta == BETAS[-1]]
x = np.arange(len(sub))
axes[0, 1].bar(x - 0.2, sub.varpi, width=0.4, label="varpi")
axes[0, 1].bar(x + 0.2, sub.baseline, width=0.4, label="baseline (rang_eff/d)")
axes[0, 1].set_xticks(x); axes[0, 1].set_xticklabels(sub.checkpoint)
axes[0, 1].set_title(f"varpi vs baseline (beta={BETAS[-1]})")
axes[0, 1].legend()

axes[1, 0].scatter(df_e2.sqrt_varpi, df_e2.alpha_tilde_star)
lims = [0, 1.05]
axes[1, 0].plot(lims, lims, "k--", lw=1)
axes[1, 0].set_xlabel("sqrt(varpi)"); axes[1, 0].set_ylabel("alpha_tilde_star")
axes[1, 0].set_title("alpha_tilde_star vs sqrt(varpi)")

for beta in BETAS:
    sub = df_e2[df_e2.beta == beta]
    axes[1, 1].plot(sub.checkpoint, sub.v_hat, marker="o", label=f"beta={beta}")
axes[1, 1].axhline(1.0, color="k", ls="--", lw=1)
axes[1, 1].set_title("v_hat = ||v||/rho sur les checkpoints")
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e2_{CONFIG_NAME}.png"), dpi=130)
plt.show()


### Verdict E2

*(a remplir apres execution, EN PRIORITE sur les valeurs `cnn`/r32p : varpi vs
baseline, v_hat aux 3 betas, Theta. Rappel : ne pas conclure sur la seule
config `linear`.)*


## E3 -- Stabilite de Gbar et cout du one-shot (BLOQUANT, tres bon marche)

Teste si une configuration unique `ubar` sert toute la trajectoire. Verifie
aussi explicitement la note de session : a `beta=0.10`, `gamma=0.3`, classes
equilibrees, `s_beta > 1` donc les plafonds par classe DEVRAIENT etre actifs
et `||u*||_1` devrait etre strictement < beta pour une attaque a une seule
classe source -- compare `capacity=True` vs `capacity=False` sur ce point precis.


In [ ]:
rows_e3 = []
u_star_by_ckpt = {}

for name in CKPT_NAMES:
    c = CACHE[name]
    Gbar, pairs, Q, pi = c["Gbar"], c["pairs"], c["Q"], c["pi"]
    v = v_of(name, E1_BETA, "identity")
    c_qp = (Gbar.T @ v).numpy().astype(np.float64)
    u_star = pl.solve_qp(Q, c_qp, E1_BETA, pi, GAMMA, pairs, scope="aggregate", capacity=True)
    u_star_by_ckpt[name] = u_star

    v_np = v.numpy().astype(np.float64)
    u_np = u_star.numpy()
    quad = float(u_np @ Q @ u_np)
    a_k = float(v_np @ v_np) - 2 * float(np.dot(u_np, c_qp)) + quad   # ||Gbar u* - v||^2
    rho_k = E1_BETA * float(Gbar.norm(dim=0).max())
    rows_e3.append(dict(checkpoint=name, a_over_rho2=a_k / (rho_k ** 2 + 1e-12),
                         l1_u_star=float(u_np.sum()), rho_k=rho_k))

df_e3_perckpt = pd.DataFrame(rows_e3)
print(df_e3_perckpt)

# --- QP couple : ubar partage --------------------------------------------
mu = {}
Q_sum, c_sum = None, None
for name in CKPT_NAMES:
    c = CACHE[name]
    Gbar, pairs, Q = c["Gbar"], c["pairs"], c["Q"]
    v = v_of(name, E1_BETA, "identity")
    c_qp = (Gbar.T @ v).numpy().astype(np.float64)
    rho_k = float(df_e3_perckpt[df_e3_perckpt.checkpoint == name].rho_k.iloc[0])
    mu_k = 1.0 / (rho_k ** 2 * len(CKPT_NAMES))
    mu[name] = mu_k
    Q_sum = mu_k * Q if Q_sum is None else Q_sum + mu_k * Q
    c_sum = mu_k * c_qp if c_sum is None else c_sum + mu_k * c_qp

ubar_shared = pl.solve_qp(Q_sum, c_sum, E1_BETA, CACHE["end"]["pi"], GAMMA,
                           CACHE["end"]["pairs"], scope="aggregate", capacity=True)

J_shared = 0.0
for name in CKPT_NAMES:
    c = CACHE[name]
    Gbar = c["Gbar"]
    v_np = v_of(name, E1_BETA, "identity").numpy().astype(np.float64)
    e = (Gbar.numpy().astype(np.float64) @ ubar_shared.numpy()) - v_np
    J_shared += mu[name] * float(e @ e)

mean_perckpt = df_e3_perckpt.a_over_rho2.mean()
gap = J_shared - mean_perckpt
gap_pct = gap / mean_perckpt * 100 if mean_perckpt != 0 else float("nan")
print(f"moyenne_k(a_k/rho_k^2) = {mean_perckpt:.5g}")
print(f"J(ubar_partage)        = {J_shared:.5g}")
print(f"ecart one-shot         = {gap:.5g} ({gap_pct:.1f}%)")

for name in CKPT_NAMES:
    r = df_e3_perckpt[df_e3_perckpt.checkpoint == name].iloc[0]
    record(CONFIG_NAME, None, name, E1_BETA, "E3", "a_over_rho2", r.a_over_rho2)
    record(CONFIG_NAME, None, name, E1_BETA, "E3", "l1_u_star", r.l1_u_star)
record(CONFIG_NAME, None, "shared", E1_BETA, "E3", "J_shared", J_shared)
record(CONFIG_NAME, None, "shared", E1_BETA, "E3", "gap_pct", gap_pct)

# --- cosinus u_k* entre checkpoints, cosinus colonne-a-colonne de Gbar_k ---
ckpt_pairs = list(itertools.combinations(CKPT_NAMES, 2))
for (n1, n2) in ckpt_pairs:
    u1, u2 = u_star_by_ckpt[n1].numpy(), u_star_by_ckpt[n2].numpy()
    denom = np.linalg.norm(u1) * np.linalg.norm(u2)
    cos_u = float(np.dot(u1, u2) / denom) if denom > 0 else float("nan")
    record(CONFIG_NAME, None, f"{n1}_vs_{n2}", E1_BETA, "E3", "cos_u_star", cos_u)
    print(f"cos(u*_{n1}, u*_{n2}) = {cos_u:.4f}")

    G1, G2 = CACHE[n1]["Gbar"].numpy(), CACHE[n2]["Gbar"].numpy()
    cos_cols = [float(np.dot(G1[:, j], G2[:, j]) /
                       (np.linalg.norm(G1[:, j]) * np.linalg.norm(G2[:, j]) + 1e-12))
                for j in range(G1.shape[1])]
    record(CONFIG_NAME, None, f"{n1}_vs_{n2}", E1_BETA, "E3", "mean_col_cos_Gbar", float(np.mean(cos_cols)))
    print(f"cos colonne moyen(Gbar_{n1}, Gbar_{n2}) = {np.mean(cos_cols):.4f}")

# --- support / budget reellement depense -----------------------------------
for name in CKPT_NAMES:
    u = u_star_by_ckpt[name].numpy()
    l1 = float(u.sum())
    frac = l1 / E1_BETA
    top = sorted(zip(CACHE[name]["pairs"], u), key=lambda t: -t[1])[:5]
    print(name, f"||u*||_1/beta = {frac:.3f}, top paires:", top)
    record(CONFIG_NAME, None, name, E1_BETA, "E3", "l1_over_beta", frac)

# --- verification explicite: s_beta > 1 => plafonds actifs a beta=0.10 -----
beta_check = 0.10
c_end = CACHE["end"]
v_chk = v_of("end", beta_check, "identity")
c_qp_chk = (c_end["Gbar"].T @ v_chk).numpy().astype(np.float64)
u_cap_true = pl.solve_qp(c_end["Q"], c_qp_chk, beta_check, c_end["pi"], GAMMA, c_end["pairs"],
                          scope="aggregate", capacity=True)
u_cap_false = pl.solve_qp(c_end["Q"], c_qp_chk, beta_check, c_end["pi"], GAMMA, c_end["pairs"],
                           scope="aggregate", capacity=False)
s_beta_check = beta_check / (GAMMA * min(c_end["pi"].values()))
print(f"\ns_beta a beta=0.10: {s_beta_check:.3f} (>1 => plafonds par classe doivent etre actifs)")
print(f"||u*||_1 capacity=True:  {float(u_cap_true.sum()):.5f} (doit etre < beta={beta_check})")
print(f"||u*||_1 capacity=False: {float(u_cap_false.sum()):.5f} (doit se rapprocher de beta={beta_check})")
record(CONFIG_NAME, None, "end", beta_check, "E3", "l1_capacity_true", float(u_cap_true.sum()))
record(CONFIG_NAME, None, "end", beta_check, "E3", "l1_capacity_false", float(u_cap_false.sum()))
record(CONFIG_NAME, None, "end", beta_check, "E3", "s_beta_check", s_beta_check)


In [ ]:
fig, axes = plt.subplots(1, len(CKPT_NAMES), figsize=(5 * len(CKPT_NAMES), 4))
for ax, name in zip(axes, CKPT_NAMES):
    u = u_star_by_ckpt[name].numpy()
    mat = np.zeros((N_CLASSES, N_CLASSES))
    for (y, z), val in zip(CACHE[name]["pairs"], u):
        mat[y, z] = val
    im = ax.imshow(mat, cmap="viridis")
    ax.set_title(f"u*_{name}"); ax.set_xlabel("z (cible)"); ax.set_ylabel("y (source)")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e3_heatmap_{CONFIG_NAME}.png"), dpi=130)
plt.show()

cos_mat = np.eye(len(CKPT_NAMES))
for i, n1 in enumerate(CKPT_NAMES):
    for j, n2 in enumerate(CKPT_NAMES):
        if i == j:
            continue
        u1, u2 = u_star_by_ckpt[n1].numpy(), u_star_by_ckpt[n2].numpy()
        denom = np.linalg.norm(u1) * np.linalg.norm(u2)
        cos_mat[i, j] = np.dot(u1, u2) / denom if denom > 0 else np.nan

fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(cos_mat, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(CKPT_NAMES))); ax.set_xticklabels(CKPT_NAMES)
ax.set_yticks(range(len(CKPT_NAMES))); ax.set_yticklabels(CKPT_NAMES)
for i in range(len(CKPT_NAMES)):
    for j in range(len(CKPT_NAMES)):
        ax.text(j, i, f"{cos_mat[i, j]:.2f}", ha="center", va="center", color="w")
plt.colorbar(im)
plt.title("cos(u*_k, u*_k')")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e3_cosmatrix_{CONFIG_NAME}.png"), dpi=130)
plt.show()

fig, ax = plt.subplots(figsize=(4, 4))
ax.bar(["moyenne par ckpt", "ubar partage"], [mean_perckpt, J_shared])
ax.set_title(f"Ecart one-shot: {gap_pct:.1f}%")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"e3_gap_{CONFIG_NAME}.png"), dpi=130)
plt.show()


### Verdict E3

*(a remplir apres execution : cosinus min entre `u*_k`, ecart one-shot en %,
resultat de la verification capacity=True vs False a beta=0.10. Conclusion
go/no-go.)*


In [ ]:
metrics_path = os.path.join(ARTIFACT_DIR, "metrics.csv")
df_metrics = pd.DataFrame(METRICS)
if os.path.exists(metrics_path):
    df_prev = pd.read_csv(metrics_path)
    df_prev = df_prev[df_prev.config != CONFIG_NAME]
    df_metrics = pd.concat([df_prev, df_metrics], ignore_index=True)
df_metrics.to_csv(metrics_path, index=False)
print(f"{len(df_metrics)} lignes -> {metrics_path}")
df_metrics.tail()


## Synthese finale

*(a remplir apres execution des deux configs `linear` et `cnn`)*

- **Verdict E1** :
- **Verdict E2** :
- **Verdict E3** :
- **Surprises** :
- **Note pour la session suivante (E4-E7)** : les agregateurs robustes du depot
  (`modules/base_utils/aggregator/`, Krum/Multi-Krum, trmean/phocas/meamed)
  operent **par tenseur de parametre** (une passe d'agregation independante par
  `p in model.parameters()`, voir `mini_train_multi`,
  `modules/base_utils/util.py` l.360-381), pas sur un vecteur de gradient
  aplati comme `Gbar`. C'est une entree directe pour la conception d'E4-E7 :
  la reponse d'un agregateur au biais `Gbar @ u` devra etre evaluee tenseur
  par tenseur, pas comme une seule direction globale.
